In [ ]:
import pandas as pd 
import numpy as np
import matplotlib as plt
import seaborn as sns
import io
from IPython.display import HTML, display
from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer, normalize

In [46]:
df = pd.read_csv('mymoviedb.csv', engine='python')

In [47]:
df

,Release_Date,Title,Overview,Popularity,Vote_Count,Vote_Average,Original_Language,Genre,Poster_Url
0,2021-12-15,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,5083.954,8940,8.3,en,"Action, Adventure, Science Fiction",https://image.tmdb.org/t/p/original/1g0dhYtq4i...
1,2022-03-01,The Batman,"In his second year of fighting crime, Batman u...",3827.658,1151,8.1,en,"Crime, Mystery, Thriller",https://image.tmdb.org/t/p/original/74xTEgt7R3...
2,2022-02-25,No Exit,Stranded at a rest stop in the mountains durin...,2618.087,122,6.3,en,Thriller,https://image.tmdb.org/t/p/original/vDHsLnOWKl...
3,2021-11-24,Encanto,"The tale of an extraordinary family, the Madri...",2402.201,5076,7.7,en,"Animation, Comedy, Family, Fantasy",https://image.tmdb.org/t/p/original/4j0PNHkMr5...
4,2021-12-22,The King's Man,As a collection of history's worst tyrants and...,1895.511,1793,7.0,en,"Action, Adventure, Thriller, War",https://image.tmdb.org/t/p/original/aq4Pwv5Xeu...
...,...,...,...,...,...,...,...,...,...
9832,1973-10-15,Badlands,A dramatization of the Starkweather-Fugate kil...,13.357,896,7.6,en,"Drama, Crime",https://image.tmdb.org/t/p/original/z81rBzHNgi...
9833,2020-10-01,Violent Delights,A female vampire falls in love with a man she ...,13.356,8,3.5,es,Horror,https://image.tmdb.org/t/p/original/4b6HY7rud6...
9834,2016-05-06,The Offering,When young and successful reporter Jamie finds...,13.355,94,5.0,en,"Mystery, Thriller, Horror",https://image.tmdb.org/t/p/original/h4uMM1wOhz...
9835,2021-03-31,The United States vs. Billie Holiday,Billie Holiday spent much of her career being ...,13.354,152,6.7,en,"Music, Drama, History",https://image.tmdb.org/t/p/original/vEzkxuE2sJ...


In [48]:
df.describe()

,Popularity
count,9827.000000
mean,40.320570
std,108.874308
min,7.100000
25%,16.127500
50%,21.191000
75%,35.174500
max,5083.954000


In [49]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9837 entries, 0 to 9836
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Release_Date       9837 non-null   str    
 1   Title              9828 non-null   str    
 2   Overview           9828 non-null   str    
 3   Popularity         9827 non-null   float64
 4   Vote_Count         9827 non-null   str    
 5   Vote_Average       9827 non-null   str    
 6   Original_Language  9827 non-null   str    
 7   Genre              9826 non-null   str    
 8   Poster_Url         9826 non-null   str    
dtypes: float64(1), str(8)
memory usage: 4.5 MB


In [50]:
df["Vote_Count"].isna().sum()

np.int64(10)

In [51]:
def read_movie_csv(path):
    text = Path(path).read_bytes().decode("utf-8", errors="replace").replace("\r", " ")
    return pd.read_csv(io.StringIO(text), engine="python")

def split_genres(value):
    return [g.strip() for g in str(value).split(",") if g.strip()]

for col in ["Popularity", "Vote_Count", "Vote_Average"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [52]:
df['Release_Date'] = pd.to_datetime(df["Release_Date"], errors="coerce")
df["Vote_Count"] = df["Vote_Count"].astype('Int64')
df["year"] = df["Release_Date"].dt.year.astype('Int64')
df["genres"] = df["Genre"].map(split_genres)
df["primary_genre"] = df["genres"].map(lambda gs: gs[0] if gs else "Unknown")

In [57]:
df['Original_Language'].value_counts()

Original_Language
en                                                                     7569
ja                                                                      645
es                                                                      339
fr                                                                      292
ko                                                                      170
zh                                                                      129
it                                                                      123
cn                                                                      109
ru                                                                       83
de                                                                       82
pt                                                                       37
da                                                                       28
hi                                                                    

In [58]:
LANGUAGE_NAMES = {
    "en": "English", "ja": "Japanese", "es": "Spanish", "fr": "French", "ko": "Korean",
    "zh": "Chinese", "cn": "Chinese", "it": "Italian", "ru": "Russian", "de": "German",
    "pt": "Portuguese", "hi": "Hindi", "da": "Danish", "no": "Norwegian", "sv": "Swedish",
    "nl": "Dutch", "th": "Thai", "pl": "Polish", "id": "Indonesian", "tr": "Turkish"
}

df["language_name"] = df["Original_Language"].map(LANGUAGE_NAMES).fillna(df["Original_Language"])

In [60]:
df.head()

,Release_Date,Title,Overview,Popularity,Vote_Count,Vote_Average,Original_Language,Genre,Poster_Url,year,genres,primary_genre,language_name
0,2021-12-15,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,5083.954,8940,8.3,en,"Action, Adventure, Science Fiction",https://image.tmdb.org/t/p/original/1g0dhYtq4i...,2021,"[Action, Adventure, Science Fiction]",Action,English
1,2022-03-01,The Batman,"In his second year of fighting crime, Batman u...",3827.658,1151,8.1,en,"Crime, Mystery, Thriller",https://image.tmdb.org/t/p/original/74xTEgt7R3...,2022,"[Crime, Mystery, Thriller]",Crime,English
2,2022-02-25,No Exit,Stranded at a rest stop in the mountains durin...,2618.087,122,6.3,en,Thriller,https://image.tmdb.org/t/p/original/vDHsLnOWKl...,2022,[Thriller],Thriller,English
3,2021-11-24,Encanto,"The tale of an extraordinary family, the Madri...",2402.201,5076,7.7,en,"Animation, Comedy, Family, Fantasy",https://image.tmdb.org/t/p/original/4j0PNHkMr5...,2021,"[Animation, Comedy, Family, Fantasy]",Animation,English
4,2021-12-22,The King's Man,As a collection of history's worst tyrants and...,1895.511,1793,7.0,en,"Action, Adventure, Thriller, War",https://image.tmdb.org/t/p/original/aq4Pwv5Xeu...,2021,"[Action, Adventure, Thriller, War]",Action,English
